In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import glob
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/iot/iot device name/network"

okui_dir = os.path.join(BASE, "okui22")
packets_dir = os.path.join(okui_dir, "packets_raw")
flows_dir = os.path.join(okui_dir, "flow_records")
features_dir = os.path.join(okui_dir, "features")

for p in [okui_dir, packets_dir, flows_dir, features_dir]:
    os.makedirs(p, exist_ok=True)

pcap_files = sorted(glob.glob(os.path.join(BASE, "normal_*.pcap")))

print("BASE exists:", os.path.exists(BASE))
print("Number of PCAPs:", len(pcap_files))
print("The first few PCAPs:", [os.path.basename(x) for x in pcap_files[:5]])

BASE exists: True
Number of PCAPs: 13
The first few PCAPs: ['normal_1.pcap', 'normal_10.pcap', 'normal_11.pcap', 'normal_12.pcap', 'normal_13.pcap']


In [16]:
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -y -qq install tshark
!tshark -v | head -n 3

!pip -q install lightgbm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Running as user "root" and group "root". This could be dangerous.
TShark (Wireshark) 3.6.2 (Git v3.6.2 packaged as 3.6.2-2)

Copyright 1998-2022 Gerald Combs <gerald@wireshark.org> and contributors.


In [ ]:
import subprocess

fields = [
    "frame.time_epoch",
    "ip.src",
    "ip.dst",
    "ip.proto",
    "tcp.srcport",
    "tcp.dstport",
    "udp.srcport",
    "udp.dstport",
    "ip.len",
    "frame.len"
]

def save_one_packet_csv(pcap_path, save_path):
    cmd = [
        "tshark",
        "-n",
        "-r", pcap_path,
        "-Y", "ip",
        "-T", "fields",
        "-E", "header=y",
        "-E", "separator=,",
        "-E", "quote=d"
    ]

    for f in fields:
        cmd += ["-e", f]

    with open(save_path, "w", encoding="utf-8", newline="") as fout:
        result = subprocess.run(
            cmd,
            stdout=fout,
            stderr=subprocess.PIPE,
            text=True
        )
    return result

for pcap in pcap_files:
    out_name = os.path.basename(pcap).replace(".pcap", "_packets.csv")
    out_path = os.path.join(packets_dir, out_name)

    if os.path.exists(out_path):
        print("Already exists, skip:", out_name)
        continue

    print("Extracting:", os.path.basename(pcap))
    result = save_one_packet_csv(pcap, out_path)

    if result.returncode != 0:
        print("Error:", os.path.basename(pcap))
        print(result.stderr[:1000])
        break
    else:
        print("save:", out_name)

print("packet csv finish")

正在提取: normal_1.pcap
保存成功: normal_1_packets.csv
正在提取: normal_10.pcap
保存成功: normal_10_packets.csv
正在提取: normal_11.pcap
保存成功: normal_11_packets.csv
正在提取: normal_12.pcap
保存成功: normal_12_packets.csv
正在提取: normal_13.pcap
保存成功: normal_13_packets.csv
正在提取: normal_2.pcap
保存成功: normal_2_packets.csv
正在提取: normal_3.pcap
保存成功: normal_3_packets.csv
正在提取: normal_4.pcap
保存成功: normal_4_packets.csv
正在提取: normal_5.pcap
保存成功: normal_5_packets.csv
正在提取: normal_6.pcap
保存成功: normal_6_packets.csv
正在提取: normal_7.pcap
保存成功: normal_7_packets.csv
正在提取: normal_8.pcap
保存成功: normal_8_packets.csv
正在提取: normal_9.pcap
保存成功: normal_9_packets.csv
packet csv 提取结束


In [ ]:
device_map = pd.DataFrame({
    "ip": [
        "192.168.1.133",
        "192.168.1.146",
        "192.168.1.152",
        "192.168.1.17",
        "192.168.1.190",
        "192.168.1.191",
        "192.168.1.192",
        "192.168.1.193",
        "192.168.1.194",
        "192.168.1.250",
        "192.168.1.30",
        "192.168.1.79",
    ],
    "best_mac": [
        "dc:56:e7:5b:61:41",
        "60:14:b3:b1:91:73",
        "00:0c:29:d2:b0:02",
        "80:3f:5d:10:17:e1",
        "00:0c:29:ee:e0:7a",
        "80:3f:5d:10:17:e1",
        "60:14:b3:b1:91:73",
        "60:14:b3:b1:91:73",
        "60:14:b3:b1:91:73",
        "d4:dc:cd:b4:26:3e",
        "80:3f:5d:10:17:e1",
        "00:c3:f4:0f:67:73",
    ],
    "service_name": [
        "service_7",
        "service_2",
        "service_1",
        "service_5",
        "service_3",
        "service_5",
        "service_2",
        "service_2",
        "service_2",
        "service_6",
        "service_5",
        "service_4",
    ]
})

map_path = os.path.join(okui_dir, "device_map.csv")
device_map.to_csv(map_path, index=False)

print("Map saved:", map_path)
device_map

map已保存: /content/drive/MyDrive/iot/iot device name/network/okui22/device_map.csv


,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


In [9]:
devmap = pd.read_csv(map_path)

devmap["ip"] = devmap["ip"].astype(str).str.strip()
devmap["best_mac"] = devmap["best_mac"].astype(str).str.strip().str.lower()
devmap["service_name"] = devmap["service_name"].astype(str).str.strip()

ip_to_mac = {}
ip_to_name = {}

for i in range(len(devmap)):
    ip = devmap.loc[i, "ip"]
    mac = devmap.loc[i, "best_mac"]
    name = devmap.loc[i, "service_name"]

    ip_to_mac[ip] = mac
    ip_to_name[ip] = name

known_ips = set(devmap["ip"].tolist())

print("known_ips:", len(known_ips))
print("known_macs:", devmap["best_mac"].nunique())
print("known_names:", devmap["service_name"].nunique())

known_ips: 12
known_macs: 7
known_names: 7


In [10]:
WINDOW = 1800.0

def clean_packet_table(df):
    df.columns = [c.strip() for c in df.columns]

    num_cols = [
        "frame.time_epoch",
        "ip.proto",
        "tcp.srcport",
        "tcp.dstport",
        "udp.srcport",
        "udp.dstport",
        "ip.len",
        "frame.len"
    ]

    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["ip.src"] = df["ip.src"].astype(str).str.strip()
    df["ip.dst"] = df["ip.dst"].astype(str).str.strip()

    df["src_port"] = df["tcp.srcport"].fillna(df["udp.srcport"])
    df["dst_port"] = df["tcp.dstport"].fillna(df["udp.dstport"])
    df["bytes_ip"] = df["ip.len"].fillna(df["frame.len"])

    df = df.dropna(subset=["frame.time_epoch", "ip.src", "ip.dst", "ip.proto", "bytes_ip"]).copy()
    df = df[(df["ip.src"] != "nan") & (df["ip.dst"] != "nan")].copy()

    return df


def keep_only_known_device_packets(df):
    mask = df["ip.src"].isin(known_ips) | df["ip.dst"].isin(known_ips)
    return df.loc[mask].copy()


def add_device_side_info(df):
    df["device_ip"] = np.where(df["ip.src"].isin(known_ips), df["ip.src"], df["ip.dst"])
    df["device_id"] = df["device_ip"].map(ip_to_mac)
    df["device_name"] = df["device_ip"].map(ip_to_name)

    df["direction"] = np.where(df["ip.src"] == df["device_ip"], "fwd", "rev")
    df["remote_ip"] = np.where(df["direction"] == "fwd", df["ip.dst"], df["ip.src"])
    df["local_port"] = np.where(df["direction"] == "fwd", df["src_port"], df["dst_port"])
    df["remote_port"] = np.where(df["direction"] == "fwd", df["dst_port"], df["src_port"])

    df = df.dropna(subset=["device_id", "device_name", "local_port", "remote_port"]).copy()

    df["local_port"] = df["local_port"].astype(int)
    df["remote_port"] = df["remote_port"].astype(int)
    df["proto"] = df["ip.proto"].astype(int)

    df["fwd_bytes"] = np.where(df["direction"] == "fwd", df["bytes_ip"], 0)
    df["rev_bytes"] = np.where(df["direction"] == "rev", df["bytes_ip"], 0)
    df["fwd_pkts"] = np.where(df["direction"] == "fwd", 1, 0)
    df["rev_pkts"] = np.where(df["direction"] == "rev", 1, 0)

    return df

In [11]:
def make_flow_table(df):
    flow = (
        df.groupby(
            ["device_id", "device_name", "device_ip", "remote_ip", "local_port", "remote_port", "proto"],
            as_index=False
        )
        .agg(
            flow_start=("frame.time_epoch", "min"),
            flow_end=("frame.time_epoch", "max"),
            octetTotalCount=("fwd_bytes", "sum"),
            reverseOctetTotalCount=("rev_bytes", "sum"),
            packetTotalCount=("fwd_pkts", "sum"),
            reversePacketTotalCount=("rev_pkts", "sum"),
        )
    )

    flow["duration"] = flow["flow_end"] - flow["flow_start"]
    flow["window_start"] = (flow["flow_end"] // WINDOW) * WINDOW

    return flow

In [12]:

def make_feature_table(flow):
    feat = (
        flow.groupby(["device_id", "device_name", "window_start"], as_index=False)
        .agg(
            record_count=("remote_ip", "size"),

            octetTotalCount_max=("octetTotalCount", "max"),
            octetTotalCount_min=("octetTotalCount", "min"),
            octetTotalCount_mean=("octetTotalCount", "mean"),
            octetTotalCount_median=("octetTotalCount", "median"),

            reverseOctetTotalCount_max=("reverseOctetTotalCount", "max"),
            reverseOctetTotalCount_min=("reverseOctetTotalCount", "min"),
            reverseOctetTotalCount_mean=("reverseOctetTotalCount", "mean"),
            reverseOctetTotalCount_median=("reverseOctetTotalCount", "median"),

            packetTotalCount_max=("packetTotalCount", "max"),
            packetTotalCount_min=("packetTotalCount", "min"),
            packetTotalCount_mean=("packetTotalCount", "mean"),
            packetTotalCount_median=("packetTotalCount", "median"),

            reversePacketTotalCount_max=("reversePacketTotalCount", "max"),
            reversePacketTotalCount_min=("reversePacketTotalCount", "min"),
            reversePacketTotalCount_mean=("reversePacketTotalCount", "mean"),
            reversePacketTotalCount_median=("reversePacketTotalCount", "median"),
        )
        .fillna(0)
    )

    return feat

In [ ]:
packet_csv_files = sorted(glob.glob(os.path.join(packets_dir, "*_packets.csv")))
all_features = []

for f in packet_csv_files:
    print("处理文件:", os.path.basename(f))

    pkt = pd.read_csv(f)
    pkt = clean_packet_table(pkt)
    pkt = keep_only_known_device_packets(pkt)

    if len(pkt) == 0:
        print("  没有匹配到已知设备IP，跳过")
        continue

    pkt = add_device_side_info(pkt)

    if len(pkt) == 0:
        print("  加完设备信息后没数据了，跳过")
        continue

    flow = make_flow_table(pkt)

    flow_save = os.path.join(
        flows_dir,
        os.path.basename(f).replace("_packets.csv", "_flows.csv")
    )
    flow.to_csv(flow_save, index=False)

    feat = make_feature_table(flow)
    feat["source_file"] = os.path.basename(f)

    first_cols = ["source_file", "device_id", "device_name", "window_start"]
    other_cols = [c for c in feat.columns if c not in first_cols]
    feat = feat[first_cols + other_cols]

    feat_save = os.path.join(
        features_dir,
        os.path.basename(f).replace("_packets.csv", "_okui22_features.csv")
    )
    feat.to_csv(feat_save, index=False)

    all_features.append(feat)

    print("  save flow:", os.path.basename(flow_save))
    print("  save feature:", os.path.basename(feat_save))

if len(all_features) > 0:
    final_df = pd.concat(all_features, ignore_index=True)
    final_path = os.path.join(features_dir, "okui22_window_features_all.csv")
    final_df.to_csv(final_path, index=False)

    print("The final summary table has been saved.:", final_path)
    print("Summary table shape:", final_df.shape)
    display(final_df.head())
else:
    print("No features were generated")

处理文件: normal_10_packets.csv


/tmp/ipykernel_32380/3710875697.py:7: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  保存flow: normal_10_flows.csv
  保存feature: normal_10_okui22_features.csv
处理文件: normal_11_packets.csv
  保存flow: normal_11_flows.csv
  保存feature: normal_11_okui22_features.csv
处理文件: normal_12_packets.csv


/tmp/ipykernel_32380/3710875697.py:7: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  保存flow: normal_12_flows.csv
  保存feature: normal_12_okui22_features.csv
处理文件: normal_13_packets.csv


...

/tmp/ipykernel_32380/3710875697.py:7: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  保存flow: normal_8_flows.csv
  保存feature: normal_8_okui22_features.csv
处理文件: normal_9_packets.csv


/tmp/ipykernel_32380/3710875697.py:7: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  pkt = pd.read_csv(f)


  保存flow: normal_9_flows.csv
  保存feature: normal_9_okui22_features.csv
最终总表保存好了: /content/drive/MyDrive/iot/iot device name/network/okui22/features/okui22_window_features_all.csv
总表shape: (367, 21)


,source_file,device_id,device_name,window_start,record_count,octetTotalCount_max,octetTotalCount_min,octetTotalCount_mean,octetTotalCount_median,reverseOctetTotalCount_max,...,reverseOctetTotalCount_mean,reverseOctetTotalCount_median,packetTotalCount_max,packetTotalCount_min,packetTotalCount_mean,packetTotalCount_median,reversePacketTotalCount_max,reversePacketTotalCount_min,reversePacketTotalCount_mean,reversePacketTotalCount_median
0,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554323e+09,94,1240.0,70.0,197.957447,186.0,1232.0,...,20.436170,0.0,18,1,2.223404,2.0,16,0,0.244681,0.0
1,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554325e+09,138,17089.0,62.0,354.891304,186.0,47043.0,...,460.376812,0.0,37,1,2.673913,2.0,35,0,0.739130,0.0
2,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554327e+09,128,2750.0,70.0,209.570312,186.0,4221.0,...,45.492188,0.0,21,1,2.242188,2.0,18,0,0.281250,0.0
3,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554329e+09,123,2700.0,73.0,222.560976,186.0,4371.0,...,68.406504,0.0,20,1,2.260163,2.0,16,0,0.235772,0.0
4,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554331e+09,127,2700.0,73.0,220.803150,186.0,5687.0,...,70.330709,0.0,22,1,2.291339,2.0,20,0,0.283465,0.0


In [ ]:
final_path = os.path.join(features_dir, "okui22_window_features_all.csv")
df = pd.read_csv(final_path)

print("shape:", df.shape)
print("\ncolumns:")
print(df.columns.tolist())

print("\nNumber of samples per device_id:")
print(df["device_id"].value_counts())

print("\nNumber of samples per device_name:")
print(df["device_name"].value_counts())

df.head()

shape: (367, 21)

columns:
['source_file', 'device_id', 'device_name', 'window_start', 'record_count', 'octetTotalCount_max', 'octetTotalCount_min', 'octetTotalCount_mean', 'octetTotalCount_median', 'reverseOctetTotalCount_max', 'reverseOctetTotalCount_min', 'reverseOctetTotalCount_mean', 'reverseOctetTotalCount_median', 'packetTotalCount_max', 'packetTotalCount_min', 'packetTotalCount_mean', 'packetTotalCount_median', 'reversePacketTotalCount_max', 'reversePacketTotalCount_min', 'reversePacketTotalCount_mean', 'reversePacketTotalCount_median']

每个device_id的样本数:
device_id
00:0c:29:d2:b0:02    94
00:0c:29:ee:e0:7a    94
00:c3:f4:0f:67:73    65
60:14:b3:b1:91:73    44
80:3f:5d:10:17:e1    28
dc:56:e7:5b:61:41    22
d4:dc:cd:b4:26:3e    20
Name: count, dtype: int64

每个device_name的样本数:
device_name
service_1    94
service_3    94
service_4    65
service_2    44
service_5    28
service_7    22
service_6    20
Name: count, dtype: int64


,source_file,device_id,device_name,window_start,record_count,octetTotalCount_max,octetTotalCount_min,octetTotalCount_mean,octetTotalCount_median,reverseOctetTotalCount_max,...,reverseOctetTotalCount_mean,reverseOctetTotalCount_median,packetTotalCount_max,packetTotalCount_min,packetTotalCount_mean,packetTotalCount_median,reversePacketTotalCount_max,reversePacketTotalCount_min,reversePacketTotalCount_mean,reversePacketTotalCount_median
0,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554323e+09,94,1240.0,70.0,197.957447,186.0,1232.0,...,20.436170,0.0,18,1,2.223404,2.0,16,0,0.244681,0.0
1,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554325e+09,138,17089.0,62.0,354.891304,186.0,47043.0,...,460.376812,0.0,37,1,2.673913,2.0,35,0,0.739130,0.0
2,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554327e+09,128,2750.0,70.0,209.570312,186.0,4221.0,...,45.492188,0.0,21,1,2.242188,2.0,18,0,0.281250,0.0
3,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554329e+09,123,2700.0,73.0,222.560976,186.0,4371.0,...,68.406504,0.0,20,1,2.260163,2.0,16,0,0.235772,0.0
4,normal_10_packets.csv,00:0c:29:d2:b0:02,service_1,1.554331e+09,127,2700.0,73.0,220.803150,186.0,5687.0,...,70.330709,0.0,22,1,2.291339,2.0,20,0,0.283465,0.0


In [ ]:
label_col = "device_id"

drop_cols = ["source_file", "device_id", "device_name", "window_start"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[label_col].copy()

print("Feature number:", len(feature_cols))
print("Feature column:")
print(feature_cols)

print("\nLabel Distribution:")
print(y.value_counts())

特征数: 17
特征列:
['record_count', 'octetTotalCount_max', 'octetTotalCount_min', 'octetTotalCount_mean', 'octetTotalCount_median', 'reverseOctetTotalCount_max', 'reverseOctetTotalCount_min', 'reverseOctetTotalCount_mean', 'reverseOctetTotalCount_median', 'packetTotalCount_max', 'packetTotalCount_min', 'packetTotalCount_mean', 'packetTotalCount_median', 'reversePacketTotalCount_max', 'reversePacketTotalCount_min', 'reversePacketTotalCount_mean', 'reversePacketTotalCount_median']

标签分布:
device_id
00:0c:29:d2:b0:02    94
00:0c:29:ee:e0:7a    94
00:c3:f4:0f:67:73    65
60:14:b3:b1:91:73    44
80:3f:5d:10:17:e1    28
dc:56:e7:5b:61:41    22
d4:dc:cd:b4:26:3e    20
Name: count, dtype: int64


In [17]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import average_precision_score

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_enc,
    test_size=0.3,
    random_state=42,
    stratify=y_enc
)

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)
print("classes:", list(le.classes_))

train shape: (256, 17)
test shape: (111, 17)
classes: ['00:0c:29:d2:b0:02', '00:0c:29:ee:e0:7a', '00:c3:f4:0f:67:73', '60:14:b3:b1:91:73', '80:3f:5d:10:17:e1', 'd4:dc:cd:b4:26:3e', 'dc:56:e7:5b:61:41']


In [18]:
clf = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

clf.fit(X_train, y_train)

proba = clf.predict_proba(X_test)
if isinstance(proba, list):
    proba = np.column_stack(proba)

y_test_bin = label_binarize(y_test, classes=np.arange(len(le.classes_)))

macro_aucpr = average_precision_score(y_test_bin, proba, average="macro")
weighted_aucpr = average_precision_score(y_test_bin, proba, average="weighted")

print("macro-AUCPR =", macro_aucpr)
print("weighted-AUCPR =", weighted_aucpr)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000242 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 520
[LightGBM] [Info] Number of data points in the train set: 256, number of used features: 13
[LightGBM] [Info] Start training from score -1.355523
[LightGBM] [Info] Start training from score -1.355523
[LightGBM] [Info] Start training from score -1.738515
[LightGBM] [Info] Start training from score -2.111190
[LightGBM] [Info] Start training from score -2.600738
[LightGBM] [Info] Start training from score -2.906120
[LightGBM] [Info] Start training from score -2.837127
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

IDLE ACTIVE MIX

In [ ]:
import os
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import average_precision_score


BASE = "/content/drive/MyDrive/iot/iot device name/network"
FEATURE_DIR = os.path.join(BASE, "Okui22", "features")
final_out = os.path.join(FEATURE_DIR, "okui22_window_features_all.csv")

df = pd.read_csv(final_out).copy()

idle_train_files = [f"normal_{i}_packets.csv" for i in range(1, 7)]   # 1-6
idle_test_files  = [f"normal_{i}_packets.csv" for i in range(7, 9)]   # 7-8

active_train_files = [f"normal_{i}_packets.csv" for i in range(9, 12)]   # 9-11
active_test_files  = [f"normal_{i}_packets.csv" for i in range(12, 14)]  # 12-13

train_files = idle_train_files + active_train_files
test_idle_files = idle_test_files
test_active_files = active_test_files
test_mix_files = test_idle_files + test_active_files


label_col = "device_id"   
drop_cols = ["source_file", "device_id", "device_name", "window_start"]
feature_cols = [c for c in df.columns if c not in drop_cols]

train_df = df[df["source_file"].isin(train_files)].copy()
test_idle_df = df[df["source_file"].isin(test_idle_files)].copy()
test_active_df = df[df["source_file"].isin(test_active_files)].copy()
test_mix_df = df[df["source_file"].isin(test_mix_files)].copy()

print("train files:", train_files)
print("test idle files:", test_idle_files)
print("test active files:", test_active_files)

print("\ntrain shape:", train_df.shape)
print("test idle shape:", test_idle_df.shape)
print("test active shape:", test_active_df.shape)
print("test mix shape:", test_mix_df.shape)

# ========= Retain only the categories shared by the training set and each test set =========
common_labels = set(train_df[label_col])
common_labels &= set(test_idle_df[label_col])
common_labels &= set(test_active_df[label_col])
common_labels &= set(test_mix_df[label_col])
common_labels = sorted(common_labels)

train_df = train_df[train_df[label_col].isin(common_labels)].copy()
test_idle_df = test_idle_df[test_idle_df[label_col].isin(common_labels)].copy()
test_active_df = test_active_df[test_active_df[label_col].isin(common_labels)].copy()
test_mix_df = test_mix_df[test_mix_df[label_col].isin(common_labels)].copy()

print("\ncommon labels:", len(common_labels))
print("train class counts:")
print(train_df[label_col].value_counts())

# ========= 5) Coding labels =========
le = LabelEncoder()
le.fit(common_labels)

train_df["y"] = le.transform(train_df[label_col])
test_idle_df["y"] = le.transform(test_idle_df[label_col])
test_active_df["y"] = le.transform(test_active_df[label_col])
test_mix_df["y"] = le.transform(test_mix_df[label_col])


X_train = train_df[feature_cols].copy()
y_train = train_df["y"].values

clf = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)
clf.fit(X_train, y_train)


def macro_aucpr_for_subset(sub_df):
    if len(sub_df) == 0:
        return np.nan

    X_sub = sub_df[feature_cols].copy()
    y_sub = sub_df["y"].values

    proba = clf.predict_proba(X_sub)
    if isinstance(proba, list):
        proba = np.column_stack(proba)

    Y_bin = label_binarize(y_sub, classes=np.arange(len(le.classes_)))

    scores = []
    for i in range(len(le.classes_)):
        if Y_bin[:, i].sum() == 0:
            continue
        ap = average_precision_score(Y_bin[:, i], proba[:, i])
        scores.append(ap)

    return float(np.mean(scores)) if len(scores) > 0 else np.nan

idle_aucpr = macro_aucpr_for_subset(test_idle_df)
active_aucpr = macro_aucpr_for_subset(test_active_df)
mix_aucpr = macro_aucpr_for_subset(test_mix_df)

result_table = pd.DataFrame([{
    "Train": "Mix",
    "Test: Idle": idle_aucpr,
    "Test: Active": active_aucpr,
    "Test: Mix": mix_aucpr,
}])

display(result_table)